<a href="https://colab.research.google.com/github/meetj6897/Develop-LLM-from-scratch/blob/main/LLM_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

tokneization and embedding

In [28]:
#getting the book from huggin face
import os
#taking the pdf from the local or huggingface
from datasets import load_dataset

ds = load_dataset("DarwinAnim8or/the-verdict")

##encode method (text to token)

In [29]:
text_book_list = ds['train']['text'][0:] # text data as a list
text_book_full = " ".join(text_book_list) # join all text into a single string

display(text_book_full[:500]) # Display the first 500 characters of the full text

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)  "The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it\''

In [30]:
#tokenize all the character in the book and then give the token id to them
#actual llm have huge amount of the data
#so to make the otkne we have different librabry in earli we have NLTK now we are usingthe re (regular expression operation)

import re

text="hello my name is meet joshi. I am AI engineer."
result=re.split(r'(\s)',text) # spliting using the white space
print(result)

result=re.split(r'([,.:;?!_"()\\]|--|\s)',text) # spliting using the white space or , .
print(result)

#remove white space character so we loopoing over each result

result=[token for token in result if token!=' ']
print("removed whilte space",result)

#sould we remove the white space ?
"""
advantage is it reduces the memory requirements, while keeping them useful if we train models which is sensitive on the exact structure of the text
ex: if we are training them on python code then white space is imp as python sensitive to indentation and spacing

"""
#aply this to text of the book
token=re.split(r'([,.:;?!_"()\\]|--|\s)',text_book_full)
token_without_space=[token for token in token if token!=' ' and token!=''] # Removed empty strings that can result from split at boundaries
print(token_without_space[:5])

['hello', ' ', 'my', ' ', 'name', ' ', 'is', ' ', 'meet', ' ', 'joshi.', ' ', 'I', ' ', 'am', ' ', 'AI', ' ', 'engineer.']
['hello', ' ', 'my', ' ', 'name', ' ', 'is', ' ', 'meet', ' ', 'joshi', '.', '', ' ', 'I', ' ', 'am', ' ', 'AI', ' ', 'engineer', '.', '']
removed whilte space ['hello', 'my', 'name', 'is', 'meet', 'joshi', '.', '', 'I', 'am', 'AI', 'engineer', '.', '']
['I', 'HAD', 'always', 'thought', 'Jack']


In [31]:
# 2nd step token to token id for that we are using the vocabulary which is alphabaetacally ordered and token id is given to them also each token gets id {token:id}
# vocab is have unique token and unique id

#so now we need list of unique token in the list which we have created
uniq=list(sorted(set(token_without_space)))
print(len(uniq), uniq[:25])

#assigning id to the token

vocab={}
for i,token in enumerate(uniq):
  vocab[token]=i
print(vocab.items())





1154 ['!', '"', "'", "'Are", "'It's", "'coming'", "'done'", "'subject", "'technique'", "'way", '(', ')', ',', '--', '.', ':', ';', '?', 'A', 'Ah', 'Among', 'And', 'Arrt', 'As', 'At']
dict_items([('!', 0), ('"', 1), ("'", 2), ("'Are", 3), ("'It's", 4), ("'coming'", 5), ("'done'", 6), ("'subject", 7), ("'technique'", 8), ("'way", 9), ('(', 10), (')', 11), (',', 12), ('--', 13), ('.', 14), (':', 15), (';', 16), ('?', 17), ('A', 18), ('Ah', 19), ('Among', 20), ('And', 21), ('Arrt', 22), ('As', 23), ('At', 24), ('Be', 25), ('Begin', 26), ('Burlington', 27), ('But', 28), ('By', 29), ('Carlo', 30), ('Chicago', 31), ('Claude', 32), ('Come', 33), ('Croft', 34), ('Destroyed', 35), ('Devonshire', 36), ("Don't", 37), ('Dubarry', 38), ('Emperors', 39), ('Florence', 40), ('For', 41), ('Gallery', 42), ('Gideon', 43), ('Gisburn', 44), ("Gisburn's", 45), ('Gisburns', 46), ('Grafton', 47), ('Greek', 48), ('Grindle', 49), ("Grindle's", 50), ('Grindles', 51), ('HAD', 52), ('Had', 53), ('Hang', 54), ('Has'

## decode method  (token to text)

 so now we map the token to token id for the input to llm but after the llm process it out put as token then we need mapping to convert the token id ot word
# reverse mapping of the vocabulary

In [35]:
import re

#making the class
class SimpleTokenizerV1:

  def __init__(self,vocab):
    """
self.str_to_int[token]: For each token from the iteration, this accesses the self.str_to_int dictionary.
This dictionary was initialized in the __init__ method with the provided vocab.
It acts as a lookup table where keys are the string tokens (e.g., 'hello', 'my', 'name') and values are their assigned integer IDs (e.g., 0, 1, 2).
[...]: The result of self.str_to_int[token] (which is the integer ID for that token) is collected for each token, forming a new list of integers. This new list is then assigned to the variable ids.
In essence, for every text token, the code looks up its pre-assigned integer ID in the self.str_to_int dictionary and compiles these IDs into a list.

What can I help you build

"""
    self.str_to_int=vocab # we are passing the vocab not text and vocab is prebuilt in above cell
    self.int_to_str={i:s for s,i in vocab.items()} # Fix: Correctly invert keys and values to map id to string

  def encoder(self,text):
    token=re.split(r'([,.:;?!_"()\\]|--|\s)',text)
    token_without_space=[token for token in token if token!=' ' and token!=''] # Removed empty strings that can result from split at boundaries
    ids=[self.str_to_int[token] for token in token_without_space] # converting the token and giving token id
    return ids

  def decoder(self,ids):
    text=" ".join([self.int_to_str[i] for i in ids])  # here we join the token
    #replace space before the specified punctuation
    text=re.sub(r'\s+([,.?!;"()\\])',r'\1',text) # Fix: \l should be \1 to reference the captured group
    return text

lets instanciate a new tokenizer object from the SimpleTokenizerV1 class and tokenize the text

In [36]:
ds['train']['text'][10][:100] # further split to 100 letter

'The desultory life of the Riviera lends itself to such purely academic speculations; and having, on '

In [37]:
tokenizer=SimpleTokenizerV1(vocab) # creating instance and passing vocab as input
text=ds['train']['text'][10][:100]
converted_id= tokenizer.encoder(text) #invoke the  encode method
print(converted_id)

# nove converting that id to text
convert_text= tokenizer.decoder(converted_id)
print(convert_text)

[110, 349, 645, 744, 1007, 98, 641, 609, 1037, 970, 826, 142, 937, 16, 175, 549, 12, 749]
The desultory life of the Riviera lends itself to such purely academic speculations; and having, on


### now what if we dont have the word in vocab as we have only  token

still there is solution for this issue apart llm train on huge data it moght not have that word

###**special context token**

In [ ]:
#now we are getting error as we dont have that word in vocab, so to resolve this we need to improve our vocab data and by training on large dataset
#still there is solution for this issue apart llm train on huge data it moght not have that word
# special context token

#handling unknown token  <|unk|> for unknown , |<endoftext>|
class SimpleTokenizerV2:

  def __init__(self,vocab):
    """
      self.str_to_int[token]: For each token from the iteration,
       this accesses the self.str_to_int dictionary."""

In [38]:
text="hello i am ai engineer"
converted_id= tokenizer.encoder(text) #invoke the  encode method
#now we are getting error as we dont have that word in vocab, so to resolve this we need to improve our vocab data and by training on large dataset
#still there is solution for this issue apart llm train on huge data it moght not have that word
# special context token
print(converted_id)

# nove converting that id to text
convert_text= tokenizer.decoder(converted_id)
print(convert_text)

KeyError: 'hello'